# Predict Glassdoor Reviews
Use trained model to predict sentiment of Glassdoor Reviews.

In [4]:
# @title Environment running
running_local = True  # @param {type:"boolean"}
if running_local:
    running_colab = running_kaggle = False
else:
    running_colab = False  # @param {type:"boolean"}
    running_kaggle = True  # @param {type:"boolean"}

In [5]:
if running_colab:
    from google.colab import drive

    drive.mount("/content/drive")

## Loading the model

In [7]:
import logging
import numpy as np
import pandas as pd
import platform
import random
import torch
import torch.nn as nn

from tqdm import tqdm
from transformers import BertTokenizer, BertModel

In [8]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler("predict_metrics.log"), logging.StreamHandler()],
)

In [ ]:
RANDOM_SEED = 103
BERTIMBAU_PATH = "neuralmind/bert-base-portuguese-cased"
BERTIMBAU_HIDDEN_SIZE = 768
TOKEN_MAX_LENGTH = 512
N_CLASSES = 3

PREDICTIONS_PATH = "."
VALIDATION_DATASET_NAME = 'validation_glassdoor_reviews.csv'
GOLDEN_DATASET_NAME = ".csv"

In [10]:
torch.manual_seed(RANDOM_SEED)

In [11]:
random.seed(RANDOM_SEED)

In [12]:
np.random.seed(RANDOM_SEED)

In [13]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"There are {torch.cuda.device_count()} GPU(s) available.")
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print("No GPU available, using the CPU instead.")
    device = torch.device("cpu")

There are 1 GPU(s) available.
Device name: NVIDIA GeForce GTX 1650


In [16]:
if running_colab:
    dataset = pd.read_csv(
        f"/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/golden_dataset/{VALIDATION_DATASET_NAME}"
    )
else:
    if running_kaggle:
        dataset = pd.read_csv(
            f"/kaggle/input/glassdoor-reviews-predicted/{VALIDATION_DATASET_NAME}"
        )
    else:
        dataset = pd.read_csv(f"./{VALIDATION_DATASET_NAME}")

In [17]:
dataset.head(2)

,review_id,company,employee_role,employee_detail,review_text,review_date,star_rating,sentiment
0,84941541,Sankhya Gestão de Negócios,Desenvolvedor,"Ex-funcionário(a), mais de um ano","Salário compatível, oportunidade de desenvolvi...",2 de mar. de 2024,5.0,1
1,84941541,Sankhya Gestão de Negócios,Desenvolvedor,"Ex-funcionário(a), mais de um ano",Não tenho o que reclamar da empresa,2 de mar. de 2024,5.0,-1


In [18]:
dataset.shape

(8212, 8)

In [19]:
dataset["sentiment"].value_counts()

sentiment
 1    4106
-1    4106
Name: count, dtype: int64

## Prediction over annotated dataset

In [21]:
tokenizer = BertTokenizer.from_pretrained(BERTIMBAU_PATH)

In [22]:
def convert_to_str(input_value):
    if isinstance(input_value, np.ndarray):
        input_str = " ".join(input_value)
    else:
        input_str = input_value

    return input_str

In [23]:
def predict_sentiment(review_text, model):
    outputs = []
    encoded_texts = tokenizer(
        review_text,
        max_length=TOKEN_MAX_LENGTH,
        add_special_tokens=True,
        return_token_type_ids=False,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt",
    )

    input_ids = encoded_texts["input_ids"].to(device)
    attention_mask = encoded_texts["attention_mask"].to(device)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        outputs.append(probabilities.cpu().numpy())

    return np.concatenate(outputs, axis=0)

### Predict reviews sentiment

In [24]:
total_iterations = len(dataset)
total_iterations

8212

In [ ]:
def predict(model, new_column):
    dataset[new_column] = pd.Series(dtype="int")

    for index, row in tqdm(
        dataset.iterrows(), total=total_iterations, desc="Processing"
    ):

        output_probabilities = predict_sentiment(
            review_text=row["review_text"], model=model
        )

        predicted_sentiment = np.argmax(output_probabilities)
        dataset.loc[index, new_column] = predicted_sentiment

        if index > 0 and index % 100 == 0:
            logging.info(f"Predicted rows: {index}/{total_iterations}")

            logging.info(

                f"Review Text: {row['review_text']};\nPredicted Sentiment: {predicted_sentiment}\n\n"
            )

    dataset[new_column] = dataset[new_column].astype(int)

#### Load models

In [26]:
class GlassdoorReviewsClassifier(nn.Module):
    def __init__(self, num_labels, classifier):
        super(GlassdoorReviewsClassifier, self).__init__()

        self.bert = BertModel.from_pretrained(BERTIMBAU_PATH)
        self.classifier = classifier

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs["last_hidden_state"][:, 0, :]
        x = self.classifier(x)
        return x

In [32]:
model = GlassdoorReviewsClassifier(
    num_labels=N_CLASSES,
    classifier=nn.Sequential(
        nn.Linear(BERTIMBAU_HIDDEN_SIZE, 300),
        nn.ReLU(),
        nn.Linear(300, 100),
        nn.ReLU(),
        nn.Linear(100, 50),
        nn.ReLU(),
        nn.Linear(50, N_CLASSES),
    ),
).to(device)

In [28]:
if running_local:
    FINETUNED_MODEL_PATH = (
        "../train_model/bertimbau-glassdoor-reviews-oversampled-freezing-v1.bin"
    )

    if platform.system() == "Windows":
        BERTIMBAU_PATH = "C:\\bert-base-portuguese-cased"
    else:
        BERTIMBAU_PATH = "/home/stevillis/bert-base-portuguese-cased"

if running_colab:
    FINETUNED_MODEL_PATH = "/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/bertimbau-glassdoor-reviews-oversampled-freezing-v1.bin"
    PREDICTIONS_PATH = "/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/report"
if running_kaggle:
    FINETUNED_MODEL_PATH = "/kaggle/input/bertimbau-glassdoor-reviews-oversampled-freezing-v1.bin/pytorch/bertimbau-glassdoor-reviews-oversampled-freezing-v1.bin/1/bertimbau-glassdoor-reviews-oversampled-freezing-v1.bin"

In [29]:
FINETUNED_MODEL_PATH

'../train_model/bertimbau-glassdoor-reviews-oversampled-freezing-v1.bin'

In [33]:
model.load_state_dict(torch.load(FINETUNED_MODEL_PATH, map_location=device))
model.eval()

GlassdoorReviewsClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [34]:
predict(
    model=model, new_column="predicted_sentiment_m1"
)  # predicted_sentiment_model_v1: m1 is the model with data augmentation and freezing bert layers

Processing:   0%|          | 0/8212 [00:00<?, ?it/s]c:\venvs\venv_sent_analysis\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Processing:   1%|          | 99/8212 [00:07<08:59, 15.03it/s] 2025-03-23 13:35:16,203 [INFO] Predicted rows: 100/8212
2025-03-23 13:35:16,203 [INFO] Review Text: Excelente empresa para se trabalhar, no quesito pessoas, benefícios, ambiente saudável;
Predicted Sentiment: 1


Processing:   2%|▏         | 199/8212 [00:13<09:00, 14.83it/s]2025-03-23 13:35:22,841 [INFO] Predicted rows: 200/8212
2025-03-23 13:35:22,841 [INFO] Review Text: Gestão bem definida, muitos benefícios, possibilidade de crescimento.;
Predicted Sentiment: 1


Processing:   4%|▎         | 299/8212 [00:20<08:43, 15.12it/s]2025-03-23 13:35:29,499 [INFO] Predicted rows: 3

In [35]:
dataset.head(3)

,review_id,company,employee_role,employee_detail,review_text,review_date,star_rating,sentiment,predicted_sentiment_m1
0,84941541,Sankhya Gestão de Negócios,Desenvolvedor,"Ex-funcionário(a), mais de um ano","Salário compatível, oportunidade de desenvolvi...",2 de mar. de 2024,5.0,1,1
1,84941541,Sankhya Gestão de Negócios,Desenvolvedor,"Ex-funcionário(a), mais de um ano",Não tenho o que reclamar da empresa,2 de mar. de 2024,5.0,-1,0
2,85370906,Sankhya Gestão de Negócios,Associate Product Owner Sênior,"Funcionário(a) atual, mais de 3 anos",Plano médico 100% gratuito para colaborador;\n...,15 de mar. de 2024,3.0,1,1


In [36]:
dataset["predicted_sentiment_m1"].value_counts()

predicted_sentiment_m1
1    4305
2    3310
0     597
Name: count, dtype: int64

#### Saving predicted validation dataset

In [38]:
dataset.to_csv(
    f"{PREDICTIONS_PATH}/{VALIDATION_DATASET_NAME}",
    index=False,
)

In [39]:
if running_kaggle:
    %cd /kaggle/working
    from IPython.display import FileLink
    FileLink(f"/kaggle/working/{VALIDATION_DATASET_NAME}")

#### Saving predicted golden dataset

In [ ]:
n_samples = 200

In [41]:
neutral_reviews = dataset[dataset["predicted_sentiment_m1"] == 0].sample(
    n=n_samples, random_state=RANDOM_SEED
)

In [42]:
positive_reviews = dataset[dataset["predicted_sentiment_m1"] == 1].sample(
    n=n_samples, random_state=RANDOM_SEED
)

In [43]:
negative_reviews = dataset[dataset["predicted_sentiment_m1"] == 2].sample(
    n=n_samples, random_state=RANDOM_SEED
)

In [44]:
golden_reviews_df = pd.concat([neutral_reviews, positive_reviews, negative_reviews])

In [47]:
golden_reviews_df.head(2)

,review_id,company,employee_role,employee_detail,review_text,review_date,star_rating,sentiment,predicted_sentiment_m1
3239,59712014,TOTVS,Consultor Comercial,"Funcionário(a) atual, mais de 3 anos",No momento ainda não identifiquei.,15 de fev. de 2022,5.0,-1,0
7461,43405389,TOTVS,Business Analyst,Funcionário(a) atual,Não vejo contras no momento,3 de mar. de 2021,4.0,-1,0


In [49]:
golden_reviews_df["predicted_sentiment_m1"].value_counts()

predicted_sentiment_m1
0    200
1    200
2    200
Name: count, dtype: int64

In [45]:
golden_reviews_df.to_csv(
    f"{PREDICTIONS_PATH}/{GOLDEN_DATASET_NAME}",
    index=False,
)

In [46]:
if running_kaggle:
    %cd /kaggle/working
    from IPython.display import FileLink
    FileLink(f"/kaggle/working/{GOLDEN_DATASET_NAME}")